In [2]:
import pandas as pd
from transformers import CLIPTokenizer, CLIPTextModel
import numpy as np
import pickle
import torch
from concurrent.futures import ThreadPoolExecutor, as_completed
import asyncio, random
from concurrent.futures import ProcessPoolExecutor
from concurrent.futures import ThreadPoolExecutor
from datetime import datetime
from sklearn.metrics.pairwise import cosine_similarity

In [3]:
claude_label = pd.read_excel(f"LLM Request\\comment_annotation_claude_fix.xlsx")
gemini_label = pd.read_excel(f"LLM Request\\comment_annotation_gemini_fix.xlsx")
openai_label = pd.read_excel(f"LLM Request\\comment_annotation_openai_final.xlsx")

In [4]:
gemini_label.shape

(5000, 10)

In [5]:
full_df = claude_label.merge(gemini_label.drop(columns=["title",'first','last','workyear from',"nationality"]),on=["artwork id"],how="outer",suffixes=("_claude","_gemini"))

In [6]:
full_df

,artwork id,title,first,last,workyear from,nationality,artistic_value_answer_claude,artistic_value_comment_claude,creativity_answer_claude,creativity_comment_claude,artistic_value_answer_gemini,artistic_value_comment_gemini,creativity_answer_gemini,creativity_comment_gemini
0,230,Buste de femme (Dora Maar),Pablo,Picasso,1939,Spanish,High,Picasso's depiction uses the interaction of pa...,Yes,Dora aesthetically stimulated Picasso in a way...,High,"Picasso's ""Buste de femme (Dora Maar)"" from 19...",Yes,"""Buste de femme (Dora Maar)"" demonstrates exce..."
1,896,Mousquetaire buste,Pablo,Picasso,1968,Spanish,High,The Mousquetaire. Buste of 1968 belongs to a m...,Yes,"These portraits represented a late flowering, ...",High,"Picasso's ""Mousquetaire buste"" from 1968 is wi...",Yes,"The ""Mousquetaire buste"" exhibits significant ..."
2,1241,Le transformateur,Pablo,Picasso,1953,Spanish,High,The Transformateur paintings serve as an index...,Yes,These paintings represent genuine creative inn...,High,"""Le transformateur"" by Pablo Picasso, created ...",Yes,Pablo Picasso's entire career is characterized...
3,1596,Le peintre et son modèle,Pablo,Picasso,1964,Spanish,High,"In Picasso's later years, the theme of painter...",Yes,The work demonstrates residual Cubism through ...,High,"Picasso's ""Le peintre et son modèle"" from 1964...",Yes,"""Le peintre et son modèle"" (1964) demonstrates..."
4,1733,Portrait de Sylvette,Pablo,Picasso,1954,Spanish,High,The Portrait de Sylvette is recognized by muse...,Yes,The Sylvette series represents the most concen...,High,"Picasso's ""Portrait de Sylvette"" from 1954 is ...",Yes,"""Portrait de Sylvette"" exemplifies creativity ..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4995,424928667,Au théatre,Pablo,Picasso,1966,Spanish,Unable to Determine,The available sources do not provide sufficien...,Unable to Determine,To properly assess creativity according to you...,High,"Picasso's 1966 artwork, ""Au théâtre,"" is consi...",Yes,"Picasso's ""Au théâtre"" demonstrates creativity..."
4996,424928669,Couverture Mourlot III,Pablo,Picasso,1956,Spanish,Limited Evidence in Sources,The work was created as the cover of the refer...,Insufficient Critical Evidence,The search results do not provide expert analy...,High,"""Couverture Mourlot III"" by Pablo Picasso, cre...",Yes,"""Couverture Mourlot III"" demonstrates signific..."
4997,424931542,Face of May,Pablo,Picasso,1946,Spanish,High,"After World War II, Picasso began working inte...",Yes,"Picasso made lithographs since the 1920s, but ...",High,"The artistic value of Pablo Picasso's ""Face of...",Yes,"""Face of May"" demonstrates creativity through ..."
4998,424934832,Fleur bleue,Pablo,Picasso,1964,Spanish,"authoritative commentary on Picasso's ""Fleur b...","To provide you with accurate, authoritative an...","for authoritative commentary on Picasso's ""Fle...","To provide you with accurate, authoritative an...",High,"Picasso's ""Fleur bleue,"" created in 1964, is r...",Yes,"""Fleur bleue"" by Pablo Picasso, created in 196..."


In [7]:
full_df = full_df.merge(openai_label.drop(columns=["title",'first','last','workyear from',"nationality"]),on=["artwork id"],how="outer",suffixes=(None,"_openai"))

In [8]:
claude_embed = np.load(f"clip_embeddings_claude_final.npy",allow_pickle=True)
gemini_embed = np.load(f"clip_embeddings_gemini_final.npy",allow_pickle=True)
openai_embed = np.load(f"clip_embeddings_openai_final.npy",allow_pickle=True)

# Golden Set

Golden set is a set of samples that have the same answers for "type" and "creative". To add to confidence, only samples with cosine similarity above a given threshold are kept.

In [9]:
consistent_creative=[]
consistent_artist=[]
consistent_overall=[]
for i in range(full_df.shape[0]):
    row = full_df.iloc[i]
    if (row.artistic_value_answer_claude == row.artistic_value_answer.strip()) & (row.artistic_value_answer.strip() == row.artistic_value_answer_gemini):
        artist_con=1
        consistent_artist.append(1)
    else:
        artist_con=0
        consistent_artist.append(0)
    if (row.creativity_answer_claude == row.creativity_answer.strip()) & (row.creativity_answer.strip() == row.creativity_answer_gemini):
        creative=1
        consistent_creative.append(1)
    else:
        creative=0
        consistent_creative.append(0)

    if artist_con+creative==2:
        consistent_overall.append(1)
    else:
        consistent_overall.append(0)

In [10]:
np.sum(consistent_artist)

np.int64(3393)

In [11]:
np.sum(consistent_creative)

np.int64(3005)

In [12]:
np.sum(consistent_overall)

np.int64(2986)

In [13]:
checking=full_df.copy()
checking["creative_consist"]=consistent_creative
checking["artistic_consist"]=consistent_artist
checking["overall_consist"]=consistent_overall

In [14]:
print(f"""
For type, there are {np.sum(checking["artistic_consist"])} samples that are consistent.
""")

print(f"""
For creative, there are {np.sum(checking["creative_consist"])} samples that are consistent.
""")


For type, there are 3393 samples that are consistent.


For creative, there are 3005 samples that are consistent.



In [15]:
print(f"""
When looking at only type and creative, there are {np.sum(checking["overall_consist"])} samples that are consistent.
""")


When looking at only type and creative, there are 2986 samples that are consistent.



In [16]:
embed_consistent_creative=[]
embed_consistent_artistic=[]
embed_consistent_overall=[]
print(f"{datetime.now():%Y-%m-%d %H:%M:%S}: Start")
for i in range(openai_embed.shape[0]):
    artistic_sim= cosine_similarity([claude_embed[i][0],gemini_embed[i][0],openai_embed[i][0]])
    if (artistic_sim > 0.65).all():
        artistic_con=1
        embed_consistent_artistic.append(1)
    else:
        artistic_con=0
        embed_consistent_artistic.append(0)
    creative_sim= cosine_similarity([claude_embed[i][1],gemini_embed[i][1],openai_embed[i][1]])
    if (creative_sim > 0.65).all():
        creative=1
        embed_consistent_creative.append(1)
    else:
        creative=0
        embed_consistent_creative.append(0)

    
    if artistic_con+creative==2:
        embed_consistent_overall.append(1)
    else:
        embed_consistent_overall.append(0)
    if i%2500==0:
        print(f"{datetime.now():%Y-%m-%d %H:%M:%S}: Currently at {i}")
print(f"{datetime.now():%Y-%m-%d %H:%M:%S}: End")

2025-12-09 02:38:30: Start
2025-12-09 02:38:30: Currently at 0
2025-12-09 02:38:31: Currently at 2500
2025-12-09 02:38:32: End


In [17]:
checking["embed_artistic_consist"]=embed_consistent_artistic
checking["embed_creative_consist"]=embed_consistent_creative
checking["embed_overall_consist"]=embed_consistent_overall

In [18]:
print(f"""
[Easy] For artistic, there are {np.sum(checking["artistic_consist"])} samples that are consistent.
[Hard] For artistic, there are {checking[(checking["artistic_consist"]==1) & (checking["embed_artistic_consist"]==1)].shape[0]} samples that are consistent.
""")

print(f"""
[Easy] For creative, there are {np.sum(checking["creative_consist"])} samples that are consistent.
[Hard] For creative, there are {checking[(checking["creative_consist"]==1) & (checking["embed_creative_consist"]==1)].shape[0]} samples that are consistent.
""")

print(f"""
[Easy]For overall, there are {np.sum(checking["overall_consist"])} samples that are consistent.
[Hard] For overall, there are {checking[(checking["overall_consist"]==1) & (checking["embed_overall_consist"]==1)].shape[0]} samples that are consistent.
""")


[Easy] For artistic, there are 3393 samples that are consistent.
[Hard] For artistic, there are 2423 samples that are consistent.


[Easy] For creative, there are 3005 samples that are consistent.
[Hard] For creative, there are 1746 samples that are consistent.


[Easy]For overall, there are 2986 samples that are consistent.
[Hard] For overall, there are 1327 samples that are consistent.



In [19]:
golden_set_move_creative = checking[(checking["creative_consist"]==1) & (checking["embed_creative_consist"]==1)]

In [20]:
golden_set_move_creative.shape

(1746, 24)

In [21]:
golden_set_move_creative.to_excel("golden_set_move_creative_65.xlsx",index=False)

# Double Check

In [22]:
golden_set_move_creative = pd.read_excel("golden_set_move_creative_65.xlsx")

In [23]:
golden_set_move_creative

,artwork id,title,first,last,workyear from,nationality,artistic_value_answer_claude,artistic_value_comment_claude,creativity_answer_claude,creativity_comment_claude,...,artistic_value_answer,artistic_value_comment,creativity_answer,creativity_comment,creative_consist,artistic_consist,overall_consist,embed_artistic_consist,embed_creative_consist,embed_overall_consist
0,1596,Le peintre et son modèle,Pablo,Picasso,1964,Spanish,High,"In Picasso's later years, the theme of painter...",Yes,The work demonstrates residual Cubism through ...,...,High,"""Le Peintre et Son Modèle"" is a significant wo...",Yes,"In ""Le Peintre et Son Modèle,"" Picasso innovat...",1,1,1,1,1,1
1,1781,Verre et citron,Pablo,Picasso,1944,Spanish,High,"According to art historian John Richardson, ""t...",Yes,The work belongs to a series of small-scale wo...,...,High,"""Verre et citron"" is a notable example of Pica...",Yes,"""Verre et citron"" exemplifies Picasso's innova...",1,1,1,1,1,1
2,2849,HOMME ASSIS,Pablo,Picasso,1969,Spanish,High,"""Homme Assis"" was painted during Picasso's mos...",Yes,Picasso's objective to paint 'nature' contrast...,...,High,"Pablo Picasso's ""Homme Assis"" (1969) is a sign...",Yes,"""Homme Assis"" exemplifies Picasso's innovative...",1,1,1,1,1,1
3,3100,"Femme assise dans un fauteuil tressé, en gris ...",Pablo,Picasso,1953,Spanish,High,This portrait of Françoise Gilot was painted i...,Yes,The 1953 portrait innovates beyond Picasso's e...,...,High,"""Femme assise dans un fauteuil tressé, en gris...",Yes,"Picasso's ""Femme assise dans un fauteuil tress...",1,1,1,0,1,0
4,3483,DEUX HIRONDELLES,Pablo,Picasso,1932,Spanish,High,"Painted on May 14, 1932 at the height of his c...",Yes,The work demonstrates creativity through surpr...,...,High,"Pablo Picasso's 1932 painting ""Deux Hirondelle...",Yes,"""Deux Hirondelles"" exemplifies Picasso's creat...",1,1,1,0,1,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1741,424922090,Visage de femme,Pablo,Picasso,1953,Spanish,High,"This ceramic work, likely depicting Jacqueline...",Yes,"Picasso was inspired by those around him, with...",...,High,"""Visage de femme"" (1953) is a glazed ceramic p...",Yes,"""Visage de femme"" demonstrates Picasso's creat...",1,1,1,1,1,1
1742,424922091,Visage d'homme,Pablo,Picasso,1953,Spanish,High,While specific scholarly critique of this 1953...,Yes,Picasso's use of the cast shadow as a pictoria...,...,High,"""Visage d'homme"" (1953) exemplifies Picasso's ...",Yes,"""Visage d'homme"" showcases Picasso's continuou...",1,1,1,1,1,1
1743,424924633,Vase aztèque aux quatre visages,Pablo,Picasso,1957,Spanish,High,Vase Aztèque aux quatre visages captures the s...,Yes,This work captures Picasso's restless need to ...,...,High,"Pablo Picasso's ""Vase Aztèque aux Quatre Visag...",Yes,"Picasso's ""Vase Aztèque aux Quatre Visages"" de...",1,1,1,1,1,1
1744,424927848,Mousquetaire,Pablo,Picasso,1969,Spanish,High,"Between 1966 and 1972, Picasso displayed porte...",Yes,"For Picasso, the musketeer signified the golde...",...,High,"Pablo Picasso's 1969 painting ""Mousquetaire"" e...",Yes,"In ""Mousquetaire,"" Picasso showcases significa...",1,1,1,1,1,1
